# Marine Biota, Water, and Sediment Monitoring Measurements from Basque Country Estuaries and Coastal Waters (1995–2014) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.q90a-wdsd/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Print metadata fields
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

This dataset contains multiple record sets, each with fields and columns. We'll enumerate all record sets, show their `@id`, and outline their available fields for exploration.

In [ ]:
# List all record sets and their fields using their `@id`
record_sets = [rs for rs in dataset.metadata.record_sets]
print(f"Number of record sets found: {len(record_sets)}\n")
for rec in record_sets:
    print(f"Record Set: {rec['@id']}")
    if 'fields' in rec:
        print("  Fields:")
        for field in rec['fields']:
            print(f"    - {field['@id']}: {field.get('name', '')}")
    else:
        print("  No fields found.")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Here, we extract each record set's data, demonstrate with the first available record set, and load all records into pandas DataFrames (one per record set).

In [ ]:
# Prepare a dictionary to hold dataframes per record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Loaded {len(records)} records.")
    else:
        print("  No records found for this record set.")
print("\nAvailable record set DataFrames:")
print(dataframes.keys())

# Example: Display columns and a preview for the first record set with data
example_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        example_record_set_id = k
        break

if example_record_set_id:
    print(f"\nColumns for {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    print("\nSample records:")
    display(dataframes[example_record_set_id].head())
else:
    print("No populated record set DataFrames to show.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Exploratory Data Analysis (EDA) example on the first non-empty record set
import numpy as np

if example_record_set_id is not None:
    df = dataframes[example_record_set_id].copy()
    
    # Try to find a numeric field automatically (float or int columns)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Selected numeric field for analysis: {numeric_field}")

        threshold = df[numeric_field].mean()  # Use mean as threshold in absence of domain knowledge
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nRecords with {numeric_field} > {threshold:.2f} (threshold = mean):")
        display(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a suitable categorical field (first non-numeric field)
        categ_cols = [c for c in df.columns if c not in numeric_cols]
        group_field = categ_cols[0] if categ_cols else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print(f"No numeric fields found in {example_record_set_id}.")
else:
    print("No available DataFrame for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Visualization of numeric field distribution and grouped means
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and (numeric_cols and group_field is not None):
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # Plot mean numeric field by group
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization not available: No record set with both numeric and categorical fields found.")

## 6. Conclusion
In this notebook, we: 
- Loaded metadata and records for the **Marine Biota, Water, and Sediment Monitoring Measurements from Basque Country Estuaries and Coastal Waters (1995–2014)** dataset using the `mlcroissant` library.
- Provided an overview of available record sets and their fields by `@id`.
- Extracted and explored the dataset using pandas, selected numeric fields for analysis, filtered and normalized data, and demonstrated how to group and aggregate by categorical variables.
- Visualized distributions and grouped means of selected fields.

This workflow can be extended further for downstream analyses, such as environmental trend analysis, domain-specific modeling, or integration with additional datasets as described in the dataset metadata and usage notes.